# HYDROGEL_PACK — Full EDA
IMC Prosperity 4 · Round 3 · Phase 2

Structure: simple → complex
1. Load & preview
2. Fair value & price trajectory
3. Spread regime
4. Book depth
5. Returns & ACF
6. OBI signal
7. Micro-price Z-score
8. Trade flow
9. Intraday patterns
10. Combined signal heatmap & strategy hypothesis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import warnings
from scipy import stats
from scipy.stats import pearsonr

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor': '#1a1a1a',
    'axes.edgecolor': '#444',
    'axes.labelcolor': '#ccc',
    'axes.titlecolor': '#fff',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'text.color': '#ccc',
    'grid.color': '#333',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'figure.dpi': 120,
    'lines.linewidth': 1.5,
})
COLORS = ['#4fc3f7', '#81c784', '#ffb74d', '#e57373', '#ce93d8']
DAY_COLORS = {'Day 0': '#4fc3f7', 'Day 1': '#81c784', 'Day 2': '#ffb74d'}

## 1. Load & Preview

In [ ]:
prices_raw = pd.read_csv('../data/prices_round_3_combined.csv', sep=';')
trades_raw = pd.read_csv('../data/trades_round_3_combined.csv', sep=';')

px = prices_raw[prices_raw['product'] == 'HYDROGEL_PACK'].copy().reset_index(drop=True)
tr = trades_raw[trades_raw['symbol'] == 'HYDROGEL_PACK'].copy().reset_index(drop=True)

# Fill missing volume levels with 0
vol_cols = ['bid_volume_1','bid_volume_2','bid_volume_3','ask_volume_1','ask_volume_2','ask_volume_3']
px[vol_cols] = px[vol_cols].fillna(0)
price_cols = ['bid_price_1','bid_price_2','bid_price_3','ask_price_1','ask_price_2','ask_price_3']
px[price_cols] = px[price_cols].fillna(np.nan)

px['spread'] = px['ask_price_1'] - px['bid_price_1']
px['delta_mid'] = px.groupby('day')['mid_price'].diff()

# Global timestamp for plotting across days
max_ts = px.groupby('day')['timestamp'].max().to_dict()
day_offsets = {0: 0, 1: max_ts[0] + 100, 2: max_ts[0] + max_ts[1] + 200}
px['global_ts'] = px['timestamp'] + px['day'].map(day_offsets)

print(f"Price rows:  {len(px):,}  |  Days: {sorted(px['day'].unique())}")
print(f"Trade rows:  {len(tr):,}")
print(f"\nPrice columns: {list(px.columns)}")
px.head(3)

## 2. Fair Value & Price Trajectory

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)

# --- Top: all 3 days on one axis (global timestamp) ---
ax = axes[0]
for day, grp in px.groupby('day'):
    ax.plot(grp['global_ts'], grp['mid_price'], color=COLORS[day], alpha=0.9, label=f'Day {day}')
ax.axhline(10000, color='#ff6b6b', lw=1.2, ls='--', label='FV = 10,000')
# Day dividers
for d in [1, 2]:
    ax.axvline(day_offsets[d], color='#666', lw=0.8, ls=':')
ax.set_title('Mid Price — All Days (Global Timeline)', fontsize=13)
ax.set_ylabel('Mid Price')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True)

# --- Bottom: per-day with rolling mean ---
ax2 = axes[1]
for day, grp in px.groupby('day'):
    roll = grp['mid_price'].rolling(50, min_periods=1).mean()
    ax2.plot(grp['timestamp'], grp['mid_price'], color=COLORS[day], alpha=0.3, lw=0.8)
    ax2.plot(grp['timestamp'], roll, color=COLORS[day], lw=2, label=f'Day {day} (roll-50)')
ax2.axhline(10000, color='#ff6b6b', lw=1.2, ls='--', label='FV = 10,000')
ax2.set_title('Mid Price — Per Day with Rolling Mean (window=50)', fontsize=13)
ax2.set_xlabel('Timestamp')
ax2.set_ylabel('Mid Price')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True)

fig.tight_layout()
plt.show()

# Per-day stats
print("\n--- Per-day mid_price stats ---")
print(px.groupby('day')['mid_price'].agg(['mean','std','min','max']).round(2))

In [ ]:
# Linear regression per day — is there any drift?
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for i, (day, grp) in enumerate(px.groupby('day')):
    ax = axes[i]
    x = grp['timestamp'].values
    y = grp['mid_price'].values
    slope, intercept, r, p, _ = stats.linregress(x, y)
    trend = intercept + slope * x
    ax.scatter(x, y, s=0.5, alpha=0.3, color=COLORS[i])
    ax.plot(x, trend, color='#ff6b6b', lw=2, label=f'slope={slope:.5f}\nR²={r**2:.3f}')
    ax.axhline(10000, color='#666', lw=0.8, ls='--')
    ax.set_title(f'Day {day} — Linear Trend', fontsize=11)
    ax.set_xlabel('Timestamp')
    ax.legend(fontsize=8)
    ax.grid(True)

axes[0].set_ylabel('Mid Price')
fig.suptitle('Intraday Linear Regression — Is There a Ramp?', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 3. Spread Regime

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Spread distribution
ax = axes[0]
spread_counts = px['spread'].value_counts().sort_index()
ax.bar(spread_counts.index.astype(str), spread_counts.values, color=COLORS[0], edgecolor='#333')
ax.set_title('Spread Distribution (all days)', fontsize=11)
ax.set_xlabel('Spread (ticks)')
ax.set_ylabel('Count')
ax.grid(True, axis='y')
for bar, v in zip(ax.patches, spread_counts.values):
    pct = v / len(px) * 100
    if pct > 1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)

# Spread over time (global)
ax2 = axes[1]
for day, grp in px.groupby('day'):
    ax2.plot(grp['global_ts'], grp['spread'], color=COLORS[day], alpha=0.5, lw=0.6, label=f'Day {day}')
ax2.set_title('Spread Over Time', fontsize=11)
ax2.set_xlabel('Global Timestamp')
ax2.set_ylabel('Spread')
ax2.legend(fontsize=8)
ax2.grid(True)

# Spread by decile of day
ax3 = axes[2]
px['decile'] = px.groupby('day')['timestamp'].transform(
    lambda x: pd.qcut(x, 10, labels=False)
)
spread_by_decile = px.groupby('decile')['spread'].mean()
ax3.bar(spread_by_decile.index, spread_by_decile.values, color=COLORS[1], edgecolor='#333')
ax3.set_title('Mean Spread by Day Decile (intraday pattern)', fontsize=11)
ax3.set_xlabel('Decile (0=open, 9=close)')
ax3.set_ylabel('Mean Spread')
ax3.set_ylim(14, 18)
ax3.grid(True, axis='y')

fig.tight_layout()
plt.show()

print(f"Spread stats: mean={px['spread'].mean():.2f}, median={px['spread'].median():.1f}, "
      f"std={px['spread'].std():.2f}, p10={px['spread'].quantile(0.1):.1f}, p90={px['spread'].quantile(0.9):.1f}")
print(f"Spread=16: {(px['spread']==16).mean()*100:.1f}% of ticks")
print(f"Spread<16: {(px['spread']<16).mean()*100:.1f}% of ticks")

## 4. Book Depth

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

level_pairs = [
    ('bid_volume_1', 'ask_volume_1', 'Level 1'),
    ('bid_volume_2', 'ask_volume_2', 'Level 2'),
    ('bid_volume_3', 'ask_volume_3', 'Level 3'),
]

# Top row: distributions
for i, (bvol, avol, label) in enumerate(level_pairs):
    ax = axes[0][i]
    nonzero_b = px[bvol][px[bvol] > 0]
    nonzero_a = px[avol][px[avol] > 0]
    bins = range(0, int(max(nonzero_b.max(), nonzero_a.max())) + 5, 2)
    ax.hist(nonzero_b, bins=bins, alpha=0.6, color='#4fc3f7', label='Bid vol', density=True)
    ax.hist(nonzero_a, bins=bins, alpha=0.6, color='#e57373', label='Ask vol', density=True)
    ax.set_title(f'{label} Volume Distribution (non-zero)', fontsize=10)
    ax.set_xlabel('Volume')
    ax.legend(fontsize=8)
    ax.grid(True, axis='y')
    fill_rate = (px[bvol] > 0).mean() * 100
    ax.text(0.05, 0.9, f'Fill rate: {fill_rate:.1f}%', transform=ax.transAxes, fontsize=8)

# Bottom row: bid vs ask scatter (symmetry check)
for i, (bvol, avol, label) in enumerate(level_pairs):
    ax = axes[1][i]
    mask = (px[bvol] > 0) & (px[avol] > 0)
    if mask.sum() > 100:
        ax.scatter(px.loc[mask, bvol], px.loc[mask, avol],
                   s=1, alpha=0.2, color=COLORS[i])
        mn = min(px.loc[mask, bvol].min(), px.loc[mask, avol].min())
        mx = max(px.loc[mask, bvol].max(), px.loc[mask, avol].max())
        ax.plot([mn, mx], [mn, mx], 'r--', lw=1, label='y=x (perfect symmetry)')
        ax.set_title(f'{label} Bid vs Ask Vol (symmetry)', fontsize=10)
        ax.set_xlabel(f'{bvol}')
        ax.set_ylabel(f'{avol}')
        ax.legend(fontsize=7)
        ax.grid(True)
    else:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{label} (sparse)', fontsize=10)

fig.suptitle('Book Depth Analysis — All Levels', fontsize=13)
fig.tight_layout()
plt.show()

# Bid/ask depth imbalance over time
px['total_bid_depth'] = px['bid_volume_1'] + px['bid_volume_2'] + px['bid_volume_3']
px['total_ask_depth'] = px['ask_volume_1'] + px['ask_volume_2'] + px['ask_volume_3']
px['depth_ratio'] = px['total_bid_depth'] / (px['total_bid_depth'] + px['total_ask_depth']).replace(0, np.nan)

fig, ax = plt.subplots(figsize=(16, 3))
ax.plot(px['global_ts'], px['depth_ratio'], color='#81c784', alpha=0.4, lw=0.5)
ax.plot(px['global_ts'], px['depth_ratio'].rolling(100).mean(), color='#81c784', lw=2, label='Roll-100 mean')
ax.axhline(0.5, color='#ff6b6b', lw=1, ls='--', label='Balanced (0.5)')
ax.set_title('Total Depth Ratio (bid / total) Over Time', fontsize=12)
ax.set_xlabel('Global Timestamp')
ax.set_ylabel('Bid Depth Ratio')
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()

## 5. Returns & ACF

In [ ]:
from statsmodels.tsa.stattools import acf

delta = px.dropna(subset=['delta_mid'])['delta_mid']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Return distribution
ax = axes[0]
bins = np.arange(delta.min() - 0.5, delta.max() + 1.5, 1)
ax.hist(delta, bins=bins, color=COLORS[0], edgecolor='#333', density=True)
x_norm = np.linspace(delta.min(), delta.max(), 200)
ax.plot(x_norm, stats.norm.pdf(x_norm, delta.mean(), delta.std()),
        color='#ff6b6b', lw=2, ls='--', label='Normal fit')
ax.set_title('Δ Mid Distribution', fontsize=11)
ax.set_xlabel('Ticks')
ax.legend(fontsize=8)
ax.grid(True, axis='y')
ax.text(0.05, 0.92, f'std={delta.std():.2f}\nskew={delta.skew():.2f}\nkurt={delta.kurtosis():.2f}',
        transform=ax.transAxes, fontsize=8, va='top')

# ACF
ax2 = axes[1]
acf_vals = acf(delta, nlags=20, fft=True)
lags = np.arange(len(acf_vals))
ax2.bar(lags[1:], acf_vals[1:], color=[COLORS[0] if v > 0 else '#e57373' for v in acf_vals[1:]], edgecolor='#333')
ci = 1.96 / np.sqrt(len(delta))
ax2.axhline(ci, color='#fff', lw=0.8, ls='--', alpha=0.5, label=f'±95% CI ({ci:.4f})')
ax2.axhline(-ci, color='#fff', lw=0.8, ls='--', alpha=0.5)
ax2.axhline(0, color='#666', lw=0.5)
ax2.set_title('ACF of Δ Mid (lags 1–20)', fontsize=11)
ax2.set_xlabel('Lag')
ax2.set_ylabel('Autocorrelation')
ax2.legend(fontsize=8)
ax2.grid(True, axis='y')
print(f"Lag-1 ACF: {acf_vals[1]:.4f}")
print("Lags 1–10:", [f'{v:.3f}' for v in acf_vals[1:11]])

# QQ plot
ax3 = axes[2]
stats.probplot(delta, dist='norm', plot=ax3)
ax3.get_lines()[0].set(color=COLORS[0], markersize=1, alpha=0.5)
ax3.get_lines()[1].set(color='#ff6b6b', lw=2)
ax3.set_title('QQ Plot of Δ Mid', fontsize=11)
ax3.grid(True)

fig.tight_layout()
plt.show()

In [ ]:
# ACF per day
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, (day, grp) in enumerate(px.dropna(subset=['delta_mid']).groupby('day')):
    ax = axes[i]
    acf_d = acf(grp['delta_mid'], nlags=15, fft=True)
    ci_d = 1.96 / np.sqrt(len(grp))
    ax.bar(range(1, len(acf_d)), acf_d[1:],
           color=[COLORS[0] if v > 0 else '#e57373' for v in acf_d[1:]], edgecolor='#333')
    ax.axhline(ci_d, color='#fff', lw=0.8, ls='--', alpha=0.4)
    ax.axhline(-ci_d, color='#fff', lw=0.8, ls='--', alpha=0.4)
    ax.axhline(0, color='#666', lw=0.5)
    ax.set_title(f'Day {day} ACF — lag-1 = {acf_d[1]:.3f}', fontsize=11)
    ax.set_xlabel('Lag')
    ax.grid(True, axis='y')
fig.suptitle('Per-Day ACF of Δ Mid', fontsize=13)
fig.tight_layout()
plt.show()

## 6. OBI Signal

In [ ]:
# Compute OBI at all levels and L1-only
px['total_bid_vol'] = px['bid_volume_1'] + px['bid_volume_2'] + px['bid_volume_3']
px['total_ask_vol'] = px['ask_volume_1'] + px['ask_volume_2'] + px['ask_volume_3']
denom = px['total_bid_vol'] + px['total_ask_vol']
px['OBI_3L'] = np.where(denom > 0, (px['total_bid_vol'] - px['total_ask_vol']) / denom, 0)

denom1 = px['bid_volume_1'] + px['ask_volume_1']
px['OBI_1L'] = np.where(denom1 > 0, (px['bid_volume_1'] - px['ask_volume_1']) / denom1, 0)

# Forward returns
for h in [1, 5, 10, 20]:
    px[f'fwd_{h}'] = px.groupby('day')['mid_price'].transform(lambda s: s.shift(-h) - s)

print(f"OBI_3L == 0: {(px['OBI_3L'] == 0).mean()*100:.1f}% of ticks")
print(f"OBI_1L == 0: {(px['OBI_1L'] == 0).mean()*100:.1f}% of ticks")
print(f"OBI_1L non-zero ticks: {(px['OBI_1L'] != 0).sum():,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# OBI_1L distribution
ax = axes[0]
vals = px['OBI_1L']
ax.hist(vals[vals < 0], bins=30, color='#e57373', alpha=0.8, label='OBI < 0 (ask-heavy)', density=True)
ax.hist(vals[vals > 0], bins=30, color='#4fc3f7', alpha=0.8, label='OBI > 0 (bid-heavy)', density=True)
ax.axvline(0, color='#fff', lw=1, ls='--')
ax.set_title('OBI L1 Distribution (non-zero only)', fontsize=11)
ax.set_xlabel('OBI_1L')
ax.legend(fontsize=9)
ax.grid(True, axis='y')

# OBI over time (scatter of non-zero ticks)
ax2 = axes[1]
mask = px['OBI_1L'] != 0
sc = ax2.scatter(px.loc[mask, 'global_ts'], px.loc[mask, 'OBI_1L'],
                  c=px.loc[mask, 'OBI_1L'], cmap='RdBu', s=8, alpha=0.6, vmin=-1, vmax=1)
plt.colorbar(sc, ax=ax2, label='OBI_1L')
ax2.axhline(0, color='#666', lw=0.5)
ax2.set_title('OBI L1 Events Over Time (non-zero only)', fontsize=11)
ax2.set_xlabel('Global Timestamp')
ax2.set_ylabel('OBI_1L')
ax2.grid(True)

fig.tight_layout()
plt.show()

In [ ]:
# OBI predictive power
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

horizons = [1, 5, 10]
for i, h in enumerate(horizons):
    ax = axes[i]
    sub = px[px['OBI_1L'] != 0].dropna(subset=[f'fwd_{h}'])
    if len(sub) < 50:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=ax.transAxes)
        continue
    slope, intercept, r, p, se = stats.linregress(sub['OBI_1L'], sub[f'fwd_{h}'])
    x_line = np.linspace(sub['OBI_1L'].min(), sub['OBI_1L'].max(), 100)
    ax.scatter(sub['OBI_1L'], sub[f'fwd_{h}'], s=4, alpha=0.3,
               c=sub['OBI_1L'], cmap='RdBu', vmin=-1, vmax=1)
    ax.plot(x_line, intercept + slope * x_line, color='#ffb74d', lw=2)
    ax.axhline(0, color='#666', lw=0.5)
    ax.axvline(0, color='#666', lw=0.5)
    ax.set_title(f'OBI_1L → fwd_{h} | β={slope:.2f} r={r:.3f} p={p:.2e}', fontsize=9)
    ax.set_xlabel('OBI_1L')
    ax.set_ylabel(f'fwd_{h} (ticks)')
    ax.grid(True)

fig.suptitle('OBI_1L Predictive Power at H=1,5,10', fontsize=13)
fig.tight_layout()
plt.show()

# Quintile bars
sub_all = px[px['OBI_1L'] != 0].dropna(subset=['fwd_1'])
sub_all['OBI_q'] = pd.qcut(sub_all['OBI_1L'], 5, labels=['Q1\n(most ask)','Q2','Q3','Q4','Q5\n(most bid)'])
quintile_means = sub_all.groupby('OBI_q', observed=True)['fwd_1'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
colors_q = ['#e57373','#ef9a9a','#aaa','#80cbc4','#4fc3f7']
ax.bar(quintile_means.index, quintile_means.values, color=colors_q, edgecolor='#333')
ax.axhline(0, color='#fff', lw=1)
ax.set_title('Mean fwd_1 by OBI_1L Quintile (non-zero events only)', fontsize=12)
ax.set_xlabel('OBI Quintile')
ax.set_ylabel('Mean fwd_ret_1 (ticks)')
ax.grid(True, axis='y')
for bar, val in zip(ax.patches, quintile_means.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05 * np.sign(val),
            f'{val:+.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Price reaction AROUND an OBI event — event study
obi_events = px[(px['OBI_1L'] != 0)].index.tolist()
window = 10
pos_events, neg_events = [], []

for idx in obi_events:
    if idx < window or idx + window >= len(px):
        continue
    day_check = px.loc[idx-window:idx+window, 'day'].nunique()
    if day_check > 1:
        continue
    snippet = px.loc[idx-window:idx+window, 'mid_price'].values
    norm = snippet - snippet[window]  # center at event
    if len(norm) == 2*window+1:
        if px.loc[idx, 'OBI_1L'] > 0:
            pos_events.append(norm)
        else:
            neg_events.append(norm)

lags_ev = np.arange(-window, window+1)
fig, ax = plt.subplots(figsize=(12, 5))

if pos_events:
    pos_arr = np.array(pos_events)
    ax.plot(lags_ev, np.mean(pos_arr, axis=0), color='#4fc3f7', lw=2.5, label=f'OBI>0 (bid-heavy, n={len(pos_events)})')
    ax.fill_between(lags_ev,
                    np.mean(pos_arr,axis=0) - np.std(pos_arr,axis=0)/np.sqrt(len(pos_arr)),
                    np.mean(pos_arr,axis=0) + np.std(pos_arr,axis=0)/np.sqrt(len(pos_arr)),
                    alpha=0.2, color='#4fc3f7')
if neg_events:
    neg_arr = np.array(neg_events)
    ax.plot(lags_ev, np.mean(neg_arr, axis=0), color='#e57373', lw=2.5, label=f'OBI<0 (ask-heavy, n={len(neg_events)})')
    ax.fill_between(lags_ev,
                    np.mean(neg_arr,axis=0) - np.std(neg_arr,axis=0)/np.sqrt(len(neg_arr)),
                    np.mean(neg_arr,axis=0) + np.std(neg_arr,axis=0)/np.sqrt(len(neg_arr)),
                    alpha=0.2, color='#e57373')

ax.axvline(0, color='#ffb74d', lw=1.5, ls='--', label='OBI event (t=0)')
ax.axhline(0, color='#666', lw=0.5)
ax.set_title('Event Study: Price Path Around OBI_1L Events (±10 ticks)', fontsize=13)
ax.set_xlabel('Ticks from event')
ax.set_ylabel('Mid price change (ticks)')
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()

## 7. Micro-Price Z-Score Signal

In [ ]:
# Compute micro-price Z-score per day
def compute_z(grp, warmup=30):
    b1 = grp['bid_price_1']
    a1 = grp['ask_price_1']
    bv1 = grp['bid_volume_1']
    av1 = grp['ask_volume_1']
    denom = bv1 + av1
    micro = np.where(denom > 0, (b1 * av1 + a1 * bv1) / denom, grp['mid_price'])
    residual = micro - grp['mid_price'].values
    res_s = pd.Series(residual, index=grp.index)
    exp_mean = res_s.expanding(min_periods=warmup).mean()
    exp_std = res_s.expanding(min_periods=warmup).std()
    Z = (res_s - exp_mean) / exp_std.replace(0, np.nan)
    return Z, pd.Series(micro, index=grp.index), res_s

Z_list, micro_list, res_list = [], [], []
for day, grp in px.groupby('day'):
    Z, micro, res = compute_z(grp)
    Z_list.append(Z)
    micro_list.append(micro)
    res_list.append(res)

px['Z'] = pd.concat(Z_list).reindex(px.index)
px['micro'] = pd.concat(micro_list).reindex(px.index)
px['residual'] = pd.concat(res_list).reindex(px.index)

valid = px.dropna(subset=['Z', 'fwd_10'])
r_z, p_z = pearsonr(valid['Z'], valid['fwd_10'])
print(f"corr(Z, fwd_10) = {r_z:.4f}, p = {p_z:.2e}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Z distribution
ax = axes[0]
z_clip = valid['Z'].clip(-5, 5)
ax.hist(z_clip, bins=60, color=COLORS[0], edgecolor='#333', density=True)
xn = np.linspace(-5, 5, 200)
ax.plot(xn, stats.norm.pdf(xn), color='#ff6b6b', lw=2, ls='--', label='Standard Normal')
ax.set_title('Z-Score Distribution (clipped ±5)', fontsize=11)
ax.set_xlabel('Z')
ax.legend(fontsize=8)
ax.grid(True, axis='y')

# Z vs fwd_10 scatter
ax2 = axes[1]
sample = valid.sample(min(3000, len(valid)), random_state=42)
ax2.scatter(sample['Z'].clip(-4, 4), sample['fwd_10'], s=3, alpha=0.3, color=COLORS[1])
slope_z, intercept_z, *_ = stats.linregress(valid['Z'].clip(-4, 4), valid['fwd_10'])
xr = np.linspace(-4, 4, 100)
ax2.plot(xr, intercept_z + slope_z * xr, color='#ffb74d', lw=2.5)
ax2.axhline(0, color='#666', lw=0.5)
ax2.axvline(0, color='#666', lw=0.5)
ax2.set_title(f'Z → fwd_10 | r={r_z:.3f} p={p_z:.1e}', fontsize=11)
ax2.set_xlabel('Z')
ax2.set_ylabel('fwd_10 (ticks)')
ax2.grid(True)

# Quintile bars
ax3 = axes[2]
valid['Z_q'] = pd.qcut(valid['Z'].clip(-4, 4), 5,
                        labels=['Q1\n(neg)','Q2','Q3','Q4','Q5\n(pos)'])
z_q_means = valid.groupby('Z_q', observed=True)['fwd_10'].mean()
colors_z = ['#e57373','#ef9a9a','#aaa','#80cbc4','#4fc3f7']
ax3.bar(z_q_means.index, z_q_means.values, color=colors_z, edgecolor='#333')
ax3.axhline(0, color='#fff', lw=1)
ax3.set_title('Mean fwd_10 by Z Quintile', fontsize=11)
ax3.set_xlabel('Z Quintile')
ax3.set_ylabel('Mean fwd_10 (ticks)')
ax3.grid(True, axis='y')
for bar, val in zip(ax3.patches, z_q_means.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02 * np.sign(val),
             f'{val:+.2f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Micro-Price Z-Score Signal', fontsize=13)
fig.tight_layout()
plt.show()

## 8. Trade Flow Analysis

In [ ]:
# Classify trades as buy or sell by joining to nearest price row
tr_h = tr.copy()
tr_h = tr_h.sort_values(['day','timestamp']).reset_index(drop=True)
px_sorted = px.sort_values(['day','timestamp'])

# Merge-asof per day
chunks = []
for day in tr_h['day'].unique():
    t_day = tr_h[tr_h['day'] == day].copy()
    p_day = px_sorted[px_sorted['day'] == day][['timestamp','bid_price_1','ask_price_1','mid_price']].copy()
    merged = pd.merge_asof(t_day, p_day, on='timestamp', direction='nearest')
    chunks.append(merged)
tr_merged = pd.concat(chunks).reset_index(drop=True)

tr_merged['is_buy'] = tr_merged['price'] >= tr_merged['ask_price_1']
tr_merged['is_sell'] = tr_merged['price'] <= tr_merged['bid_price_1']
tr_merged['classified'] = tr_merged['is_buy'] | tr_merged['is_sell']

# Forward returns for trades
tr_merged['global_ts'] = tr_merged['timestamp'] + tr_merged['day'].map(day_offsets)
for h in [1, 5, 10]:
    tr_merged[f'fwd_{h}'] = np.nan
    for day, tgrp in tr_merged.groupby('day'):
        px_day = px[px['day'] == day].set_index('timestamp')['mid_price']
        for idx, row in tgrp.iterrows():
            ts = row['timestamp']
            future = px_day[px_day.index > ts].iloc[h-1:h]
            if not future.empty:
                tr_merged.loc[idx, f'fwd_{h}'] = future.values[0] - row['mid_price']

print(f"Total trades: {len(tr_merged)}")
print(f"Classified: {tr_merged['classified'].sum()} ({tr_merged['classified'].mean()*100:.0f}%)")
print(f"Buys: {tr_merged['is_buy'].sum()} | Sells: {tr_merged['is_sell'].sum()}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Trade prices over time
ax = axes[0][0]
buys = tr_merged[tr_merged['is_buy']]
sells = tr_merged[tr_merged['is_sell']]
ax.scatter(buys['global_ts'], buys['price'], s=10, color='#4fc3f7', alpha=0.7, label=f'Buy ({len(buys)})', zorder=3)
ax.scatter(sells['global_ts'], sells['price'], s=10, color='#e57373', alpha=0.7, label=f'Sell ({len(sells)})', zorder=3, marker='v')
ax.plot(px['global_ts'], px['mid_price'], color='#555', lw=0.5, alpha=0.5, label='Mid')
ax.set_title('Trade Locations vs Mid Price', fontsize=10)
ax.set_xlabel('Global Timestamp')
ax.set_ylabel('Price')
ax.legend(fontsize=8)
ax.grid(True)

# Trade quantity distribution
ax2 = axes[0][1]
q_counts = tr_merged['quantity'].value_counts().sort_index()
ax2.bar(q_counts.index.astype(str), q_counts.values, color=COLORS[1], edgecolor='#333')
ax2.set_title('Trade Quantity Distribution', fontsize=10)
ax2.set_xlabel('Quantity')
ax2.set_ylabel('Count')
ax2.grid(True, axis='y')

# Trades per day
ax3 = axes[0][2]
tpd = tr_merged.groupby('day').size()
ax3.bar(tpd.index, tpd.values, color=COLORS[2], edgecolor='#333')
ax3.set_title('Trades Per Day', fontsize=10)
ax3.set_xlabel('Day')
ax3.set_ylabel('Count')
ax3.grid(True, axis='y')
for bar, val in zip(ax3.patches, tpd.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', fontsize=9)

# fwd_10 after buy vs sell
ax4 = axes[1][0]
buy_fwd = buys.dropna(subset=['fwd_10'])['fwd_10']
sell_fwd = sells.dropna(subset=['fwd_10'])['fwd_10']
bins_f = np.linspace(-30, 30, 40)
ax4.hist(buy_fwd.clip(-30, 30), bins=bins_f, alpha=0.7, color='#4fc3f7', label=f'After Buy (n={len(buy_fwd)})', density=True)
ax4.hist(sell_fwd.clip(-30, 30), bins=bins_f, alpha=0.7, color='#e57373', label=f'After Sell (n={len(sell_fwd)})', density=True)
ax4.axvline(buy_fwd.mean(), color='#4fc3f7', lw=2, ls='--', label=f'Buy mean={buy_fwd.mean():+.2f}')
ax4.axvline(sell_fwd.mean(), color='#e57373', lw=2, ls='--', label=f'Sell mean={sell_fwd.mean():+.2f}')
ax4.set_title('fwd_10 After Buy vs Sell Trades', fontsize=10)
ax4.set_xlabel('fwd_10 (ticks)')
ax4.legend(fontsize=7)
ax4.grid(True, axis='y')

# t-tests
if len(buy_fwd) > 10:
    t_buy, p_buy = stats.ttest_1samp(buy_fwd, 0)
    ax4.text(0.05, 0.15, f'Buy t={t_buy:.2f} p={p_buy:.3f}', transform=ax4.transAxes, fontsize=8)
if len(sell_fwd) > 10:
    t_sell, p_sell = stats.ttest_1samp(sell_fwd, 0)
    ax4.text(0.05, 0.08, f'Sell t={t_sell:.2f} p={p_sell:.3f}', transform=ax4.transAxes, fontsize=8)

# Trade inter-arrival
ax5 = axes[1][1]
for day, grp in tr_merged.groupby('day'):
    intervals = grp['timestamp'].diff().dropna()
    ax5.hist(intervals.clip(0, 10000), bins=40, alpha=0.6, label=f'Day {day}', density=True)
ax5.set_title('Trade Inter-Arrival Time (ticks)', fontsize=10)
ax5.set_xlabel('Ticks between trades')
ax5.legend(fontsize=8)
ax5.grid(True, axis='y')

# Trade price vs mid (slippage)
ax6 = axes[1][2]
tr_merged['price_vs_mid'] = tr_merged['price'] - tr_merged['mid_price']
ax6.hist(tr_merged[tr_merged['is_buy']]['price_vs_mid'], bins=30, alpha=0.7,
          color='#4fc3f7', label='Buy (vs mid)', density=True)
ax6.hist(tr_merged[tr_merged['is_sell']]['price_vs_mid'], bins=30, alpha=0.7,
          color='#e57373', label='Sell (vs mid)', density=True)
ax6.axvline(0, color='#fff', lw=1)
ax6.set_title('Trade Price Relative to Mid', fontsize=10)
ax6.set_xlabel('Price - Mid (ticks)')
ax6.legend(fontsize=8)
ax6.grid(True, axis='y')

fig.suptitle('Trade Flow Analysis', fontsize=13)
fig.tight_layout()
plt.show()

## 9. Intraday Patterns

In [ ]:
# 10-block intraday structure
px['intraday_block'] = px.groupby('day')['timestamp'].transform(
    lambda x: pd.qcut(x, 10, labels=False)
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Mean mid by block per day
ax = axes[0][0]
for day, grp in px.groupby('day'):
    block_means = grp.groupby('intraday_block')['mid_price'].mean()
    ax.plot(block_means.index, block_means.values, marker='o', color=COLORS[day], label=f'Day {day}', lw=2)
ax.set_title('Mean Mid Price by Intraday Block (0=open, 9=close)', fontsize=11)
ax.set_xlabel('Block')
ax.set_ylabel('Mean Mid Price')
ax.legend(fontsize=9)
ax.grid(True)

# Volatility (std of delta_mid) by block
ax2 = axes[0][1]
for day, grp in px.groupby('day'):
    block_vol = grp.groupby('intraday_block')['delta_mid'].std()
    ax2.plot(block_vol.index, block_vol.values, marker='o', color=COLORS[day], label=f'Day {day}', lw=2)
ax2.set_title('Intraday Volatility (std Δ mid) by Block', fontsize=11)
ax2.set_xlabel('Block')
ax2.set_ylabel('Std of Δ Mid')
ax2.legend(fontsize=9)
ax2.grid(True)

# Mean spread by block
ax3 = axes[1][0]
for day, grp in px.groupby('day'):
    block_spread = grp.groupby('intraday_block')['spread'].mean()
    ax3.plot(block_spread.index, block_spread.values, marker='o', color=COLORS[day], label=f'Day {day}', lw=2)
ax3.set_title('Mean Spread by Intraday Block', fontsize=11)
ax3.set_xlabel('Block')
ax3.set_ylabel('Mean Spread')
ax3.legend(fontsize=9)
ax3.grid(True)

# OBI event frequency by block
ax4 = axes[1][1]
obi_events_df = px[px['OBI_1L'] != 0]
for day, grp in px.groupby('day'):
    total_per_block = grp.groupby('intraday_block').size()
    obi_per_block = obi_events_df[obi_events_df['day'] == day].groupby('intraday_block').size().reindex(total_per_block.index, fill_value=0)
    freq = obi_per_block / total_per_block
    ax4.plot(freq.index, freq.values, marker='o', color=COLORS[day], label=f'Day {day}', lw=2)
ax4.set_title('OBI Event Frequency by Block (% of ticks)', fontsize=11)
ax4.set_xlabel('Block')
ax4.set_ylabel('Fraction of ticks with OBI≠0')
ax4.legend(fontsize=9)
ax4.grid(True)

fig.suptitle('Intraday Patterns (10 Equal Blocks)', fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Rolling z-score of mid_price (stationarity visualised)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
roll_mean = px.groupby('day')['mid_price'].transform(lambda s: s.expanding().mean())
roll_std = px.groupby('day')['mid_price'].transform(lambda s: s.expanding().std())
zscore_mid = (px['mid_price'] - roll_mean) / roll_std.replace(0, np.nan)
for day, grp in px.groupby('day'):
    mask = px['day'] == day
    ax.plot(grp['timestamp'], zscore_mid[mask], color=COLORS[day], alpha=0.6, lw=0.7, label=f'Day {day}')
ax.axhline(0, color='#ff6b6b', lw=1, ls='--')
ax.axhline(2, color='#666', lw=0.5, ls=':')
ax.axhline(-2, color='#666', lw=0.5, ls=':')
ax.set_title('Rolling Z-Score of Mid Price (expanding window, per day)', fontsize=11)
ax.set_xlabel('Timestamp')
ax.set_ylabel('Z-score')
ax.legend(fontsize=8)
ax.grid(True)

# Heatmap: mid price by day × intraday block
ax2 = axes[1]
pivot = px.pivot_table(values='mid_price', index='day', columns='intraday_block', aggfunc='mean')
im = ax2.imshow(pivot.values, aspect='auto', cmap='RdYlGn', interpolation='nearest')
plt.colorbar(im, ax=ax2, label='Mean Mid Price')
ax2.set_xticks(range(10))
ax2.set_xticklabels([f'B{i}' for i in range(10)])
ax2.set_yticks(range(len(pivot.index)))
ax2.set_yticklabels([f'Day {d}' for d in pivot.index])
ax2.set_title('Mid Price Heatmap: Day × Intraday Block', fontsize=11)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax2.text(j, i, f'{pivot.values[i,j]:.0f}', ha='center', va='center', fontsize=7, color='black')

fig.tight_layout()
plt.show()

## 10. Combined Signal Dashboard & Strategy Hypothesis

In [ ]:
# Signal strength summary
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# OBI vs Z joint heatmap
ax = axes[0]
valid2 = px.dropna(subset=['Z', 'OBI_1L', 'fwd_1'])
valid2 = valid2[valid2['OBI_1L'] != 0]  # OBI events only
if len(valid2) > 0:
    valid2['Z_bin'] = pd.cut(valid2['Z'].clip(-3, 3), bins=[-3,-1,0,1,3], labels=['Z<-1','-1≤Z<0','0≤Z<1','Z>1'])
    valid2['OBI_bin'] = valid2['OBI_1L'].apply(lambda x: 'OBI>0' if x > 0 else 'OBI<0')
    heat = valid2.pivot_table(values='fwd_1', index='OBI_bin', columns='Z_bin', aggfunc='mean')
    if not heat.empty:
        im2 = ax.imshow(heat.values, aspect='auto', cmap='RdYlGn', vmin=-6, vmax=6)
        plt.colorbar(im2, ax=ax, label='Mean fwd_1')
        ax.set_xticks(range(len(heat.columns)))
        ax.set_xticklabels(heat.columns, fontsize=8)
        ax.set_yticks(range(len(heat.index)))
        ax.set_yticklabels(heat.index, fontsize=8)
        ax.set_title('Mean fwd_1: OBI × Z (OBI events only)', fontsize=10)
        for i in range(heat.shape[0]):
            for j in range(heat.shape[1]):
                v = heat.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9, color='black')

# Signal summary bar chart
ax2 = axes[1]
signal_names = ['ACF lag-1', 'OBI_1L (β)', 'Micro-Z (r)', 'Buy flow (t)', 'Sell flow (t)']
signal_values = []

# ACF lag-1
from statsmodels.tsa.stattools import acf as acf_fn
signal_values.append(acf_fn(px.dropna(subset=['delta_mid'])['delta_mid'], nlags=1, fft=True)[1])

# OBI beta normalised
sub_obi = px[px['OBI_1L'] != 0].dropna(subset=['fwd_1'])
if len(sub_obi) > 10:
    b, *_ = stats.linregress(sub_obi['OBI_1L'], sub_obi['fwd_1'])
    signal_values.append(b / 20)  # scale to [-1,1]
else:
    signal_values.append(0)

# Micro-Z correlation
signal_values.append(r_z)

# Buy/sell t-stats (normalised)
buy_f = buys.dropna(subset=['fwd_10'])['fwd_10']
sell_f = sells.dropna(subset=['fwd_10'])['fwd_10']
t_b = stats.ttest_1samp(buy_f, 0)[0] / 15 if len(buy_f) > 5 else 0
t_s = stats.ttest_1samp(sell_f, 0)[0] / 15 if len(sell_f) > 5 else 0
signal_values.extend([t_b, t_s])

bar_colors = ['#e57373' if v < 0 else '#4fc3f7' for v in signal_values]
ax2.barh(signal_names, signal_values, color=bar_colors, edgecolor='#333')
ax2.axvline(0, color='#fff', lw=1)
ax2.set_title('Normalised Signal Strength Summary', fontsize=10)
ax2.set_xlabel('Normalised value (direction + magnitude)')
ax2.grid(True, axis='x')
for i, v in enumerate(signal_values):
    ax2.text(v + 0.005 * np.sign(v), i, f'{v:+.3f}', va='center', fontsize=8)

# PnL simulation: fixed FV MM vs buy-and-hold
ax3 = axes[2]
# Simple sim: earn half-spread on each fill (stylised)
mid_series = px[px['day'] == 0]['mid_price'].values
bh_pnl = np.cumsum(np.diff(mid_series)) * 80  # max position
ax3.plot(bh_pnl, color='#aaa', lw=1, label='Buy & Hold (+80 pos)')
ax3.axhline(0, color='#666', lw=0.5)
ax3.set_title('Buy & Hold PnL (Day 0, pos=80)\n(reference: stationary → ~0)', fontsize=9)
ax3.set_xlabel('Tick')
ax3.set_ylabel('Cumulative PnL (ticks)')
ax3.legend(fontsize=8)
ax3.grid(True)

fig.tight_layout()
plt.show()

In [ ]:
# Final hypothesis printout
from statsmodels.tsa.stattools import adfuller

adf_p = adfuller(px['mid_price'].dropna(), autolag='AIC')[1]
acf1 = acf_fn(px.dropna(subset=['delta_mid'])['delta_mid'], nlags=1, fft=True)[1]
sub_obi_final = px[px['OBI_1L'] != 0].dropna(subset=['fwd_1'])
obi_beta = stats.linregress(sub_obi_final['OBI_1L'], sub_obi_final['fwd_1'])[0] if len(sub_obi_final) > 10 else 0
obi_r = pearsonr(sub_obi_final['OBI_1L'], sub_obi_final['fwd_1'])[0] if len(sub_obi_final) > 10 else 0

print("")
print("=" * 60)
print("  HYDROGEL_PACK — STRATEGY HYPOTHESIS")
print("=" * 60)
print(f"  Fair value type   : FIXED (~10,000) | ADF p={adf_p:.4f}")
print(f"  FV estimate       : {px['mid_price'].mean():.1f} | Day drift: {px.groupby('day')['mid_price'].mean().values}")
print(f"  Spread regime     : STABLE 16 ticks ({(px['spread']==16).mean()*100:.0f}% of ticks)")
print(f"  Tick volatility   : {px['delta_mid'].std():.2f} ticks (std Δ mid)")
print(f"  ACF lag-1         : {acf1:.3f} (weak mean-reversion)")
print(f"  OBI signal        : DIRECTIONAL | fires {(px['OBI_1L']!=0).mean()*100:.1f}% of ticks")
print(f"                      β={obi_beta:.2f}, r={obi_r:.3f}")
print(f"  Micro-Z signal    : WEAK MOMENTUM | r={r_z:.3f}")
print(f"  Buy-trade signal  : NOT INFORMED")
print(f"  Sell-trade signal : NOT INFORMED")
print(f"  Intraday ramp     : NONE (stationary within day)")
print("")
print("  RECOMMENDED APPROACH:")
print("  → Fixed-FV market-making at FV=10,000")
print("  → Inventory skew: shift quotes when |pos| > 40")
print("  → OBI tilt: when OBI_1L != 0, pull quote on losing side")
print("  → DO NOT use Holt smoothing (no trend)")
print("  → DO NOT use market orders (16-tick spread)")
print("  → Symmetric long/short acceptable (no directional bias)")
print("  → Confidence: HIGH")
print("=" * 60)